## Probing and Visualizing Self-Supervised Vision Transformers

---

##  Objectives

By the end of this notebook, you will be able to:

1. Load and use a pretrained **DINOv2** model via HuggingFace `transformers`
2. Extract and **visualize self-attention maps** to understand what the model focuses on
3. Explore the **patch embedding space** using PCA and discover emergent semantic segmentation
4. Evaluate feature quality via **linear probing** on a small dataset
5. **Compare** self-supervised (DINOv2) vs supervised (ViT) features — and understand *why* DINO generalizes better

---

## Background

In previous notebooks, you implemented:
- The **core self-attention mechanism** from scratch
- A full **Vision Transformer (ViT)** with patch embeddings and positional encoding
- Finetune a Swin transformer using the huggingface ecosystem

Now we ask a different question: **How good are self-supervised models and what do these models actually learn?**

**DINOv2** (Oquab et al., 2023) is a self-supervised ViT trained with a knowledge-distillation approach — no labels required. It produces remarkably rich visual features that encode semantic structure, object boundaries, and scene geometry — all without ever seeing a single annotation.

> **Key insight**: A model trained to be consistent across views of the same image implicitly learns to segment objects, match semantically similar regions, and build spatially aware representations.



---
# Section 1 — Setup & Model Loading

We will use `facebook/dinov2-small`, which has **21M parameters** — a comfortable fit for Colab's T4 GPU.

| Model variant | Params | Notes |
|---|---|---|
| dinov2-small | 21M | **We use this** |
| dinov2-base  | 86M | Fits on T4 |
| dinov2-large | 307M | Needs A100 |
| dinov2-giant | 1.1B | Cloud/cluster only |

In [ ]:
# Install dependencies
!pip install -q transformers timm umap-learn matplotlib scikit-learn Pillow requests

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import requests
from io import BytesIO

from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load DINOv2-small via HuggingFace
MODEL_NAME = "facebook/dinov2-small"

print(f"Loading processor and model: {MODEL_NAME}")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model = model.to(DEVICE)
model.eval()

# Model summary
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\nModel loaded successfully!")
print(f"  Parameters: {total_params:.1f}M")
print(f"  Patch size: {model.config.patch_size}px")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Attention heads: {model.config.num_attention_heads}")
print(f"  Transformer layers: {model.config.num_hidden_layers}")

In [ ]:
import torchvision
from torchvision.datasets import STL10

# Download STL-10 early — it will be reused in Section 4
print("Downloading STL-10 (will be reused in Section 4)...")
raw_dataset = STL10(root="./data", split="test", download=True)

STL_CLASSES = ["airplane", "bird", "car", "cat", "deer",
               "dog", "horse", "monkey", "ship", "truck"]

# Pick the first occurrence of 4 visually distinct classes
SHOW = {"dog": 5, "cat": 3, "car": 2, "bird": 1}  # name → class index

images = {}
for img_pil, label in raw_dataset:
    for name, idx in SHOW.items():
        if label == idx and name not in images:
            images[name] = img_pil
    if len(images) == len(SHOW):
        break

print(f"  ✅ Loaded {len(images)} images: {list(images.keys())}")

# Display
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.axis('off')
plt.suptitle("Our test images (from STL-10)", fontsize=15)
plt.tight_layout()
plt.show()

---
# Section 2 — Attention Map Visualization

## What are we visualizing?

In a ViT, every patch attends to every other patch. The **[CLS] token** is a special learnable token whose attention pattern reveals what the model considers most relevant when building its global image representation.

Formally, for the last Transformer layer with $H$ heads:

$$A^{(h)} \in \mathbb{R}^{(N+1) \times (N+1)}$$

where $N$ is the number of patches. We extract the row corresponding to [CLS], reshape it to a spatial grid, and visualize it as a heatmap.

### **Question 2.1**
> Before running the cell below, **write your prediction**: which regions of each image do you expect the model to attend to? Why?

In [ ]:
# ── Helper: extract CLS attention maps from last layer ───────────────────────
def get_attention_maps(image: Image.Image, model, processor, device):
    """
    Returns:
        attn_maps : np.ndarray of shape (num_heads, h_patches, w_patches)
        grid_size : (h_patches, w_patches)
    """
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # Last layer attention: shape (1, num_heads, seq_len, seq_len)
    # seq_len = 1 (CLS) + num_patches
    last_attn = outputs.attentions[-1].squeeze(0).cpu().numpy()  # (H, seq, seq)

    # CLS token attends to all patches: index 0 attends to indices 1..
    cls_attn = last_attn[:, 0, 1:]  # (num_heads, num_patches)

    # Compute grid size from patch count
    patch_size = model.config.patch_size
    h, w = inputs["pixel_values"].shape[-2:]
    h_patches, w_patches = h // patch_size, w // patch_size

    # Reshape to spatial grid
    attn_maps = cls_attn.reshape(-1, h_patches, w_patches)  # (H, h_p, w_p)

    return attn_maps, (h_patches, w_patches)

In [ ]:
# Visualize attention maps for one image
IMAGE_KEY = "cat"   # ← Change this to explore other images
image = images[IMAGE_KEY]

attn_maps, (h_patches, w_patches) = get_attention_maps(image, model, processor, DEVICE)
num_heads = attn_maps.shape[0]

print(f"Image: '{IMAGE_KEY}'")
print(f"Grid: {h_patches}×{w_patches} patches → {h_patches * w_patches} total patches")
print(f"Attention heads: {num_heads}")

# Plot all heads
fig, axes = plt.subplots(2, num_heads // 2 + 1, figsize=(20, 7))
axes = axes.flatten()

# Original image in first slot
axes[0].imshow(image)
axes[0].set_title("Original", fontsize=12, fontweight='bold')
axes[0].axis('off')

for h in range(num_heads):
    ax = axes[h + 1]
    attn = attn_maps[h]
    # Normalize per head
    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    ax.imshow(image)
    ax.imshow(attn, cmap='gray', alpha=0.6,
              extent=[0, image.width, image.height, 0])
    ax.set_title(f"Head {h+1}", fontsize=10)
    ax.axis('off')

# Hide unused axes
for ax in axes[num_heads + 1:]:
    ax.set_visible(False)

plt.suptitle(f"[CLS] Self-Attention Maps — Last Layer — '{IMAGE_KEY}'",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mean attention across all heads — often gives the clearest segmentation
mean_attn = attn_maps.mean(axis=0)
mean_attn = (mean_attn - mean_attn.min()) / (mean_attn.max() - mean_attn.min())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(image)
axes[0].set_title("Original image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mean_attn, cmap='gray')
axes[1].set_title("Mean attention map", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(image)
axes[2].imshow(mean_attn, cmap='gray', alpha=0.8,
               extent=[0, image.width, image.height, 0])
axes[2].set_title("Overlay", fontweight='bold')
axes[2].axis('off')

plt.suptitle("Mean [CLS] Attention — Emergent Object Localization",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNotice: despite no segmentation labels during training,")
print("   the model has learned to focus on the semantic object!")

### Exercise 2.1 — Layer-wise Attention Evolution

The code above only shows the **last** Transformer layer. But how does attention evolve through the network?

**Task:** Modify `get_attention_maps()` to accept a `layer_idx` parameter and visualize the mean [CLS] attention at layers 1, 4, 8, and 12 (last) for the same image.

**Expected finding:** Early layers attend locally (nearby patches); later layers capture global semantic structure.

```python
# Your code here
def get_attention_maps_at_layer(image, model, processor, device, layer_idx=-1):
    # TODO: implement
    pass

# Visualize for layers [1, 4, 8, 12]
```

In [ ]:
# ── Your solution here ───────────────────────────────────────────────────────


---
# Section 3 — Exploring the Patch Embedding Space

## Emergent Segmentation via PCA

Beyond the [CLS] token, DINOv2 produces a **feature vector per patch**. These patch-level features encode local semantic content. When we apply **PCA** to them, a remarkable phenomenon occurs:

> The first few principal components spontaneously encode **object identity**, **depth**, and **semantic regions** — without any segmentation supervision.

This is one of the most striking empirical results from the original DINO paper (Caron et al., 2021), later amplified in DINOv2.

### How it works

1. Extract patch features: shape `(N_patches, hidden_dim)` — e.g., `(256, 384)` for dinov2-small
2. Apply PCA → keep first 3 components
3. Map components to RGB channels
4. Reshape to spatial grid and display

### **Guided Question 3.1**
> Why would the first PCA component correspond to object boundaries? What property of the training objective encourages this?

In [ ]:
# ── Helper: extract patch-level features ─────────────────────────────────────
def extract_patch_features(image: Image.Image, model, processor, device):
    """
    Returns:
        features : np.ndarray of shape (h_patches, w_patches, hidden_dim)
        grid_size : (h_patches, w_patches)
    """
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # last_hidden_state: (1, 1+N_patches, hidden_dim)
    # Index 0 is [CLS], rest are patch tokens
    patch_tokens = outputs.last_hidden_state[0, 1:, :]  # (N, D)
    patch_tokens = patch_tokens.cpu().numpy()

    patch_size = model.config.patch_size
    h, w = inputs["pixel_values"].shape[-2:]
    h_patches, w_patches = h // patch_size, w // patch_size

    features = patch_tokens.reshape(h_patches, w_patches, -1)
    return features, (h_patches, w_patches)


# ── PCA visualization ─────────────────────────────────────────────────────────
def pca_rgb_visualization(features_2d: np.ndarray, n_components: int = 3):
    """
    Apply PCA to patch features and return an RGB image.
    features_2d: (h_patches, w_patches, hidden_dim)
    """
    h, w, d = features_2d.shape
    flat = features_2d.reshape(-1, d)  # (N, D)

    pca = PCA(n_components=n_components)
    reduced = pca.fit_transform(flat)  # (N, 3)

    # Normalize to [0, 1] per component
    for i in range(n_components):
        reduced[:, i] = (reduced[:, i] - reduced[:, i].min()) / \
                        (reduced[:, i].max() - reduced[:, i].min() + 1e-8)

    rgb = reduced.reshape(h, w, n_components)
    explained = pca.explained_variance_ratio_
    return rgb, explained


print("Helper functions defined.")

In [ ]:
# PCA visualization for a single image
IMAGE_KEY = "cat"   # ← Explore different images!
image = images[IMAGE_KEY]

features, grid = extract_patch_features(image, model, processor, DEVICE)
rgb_pca, explained = pca_rgb_visualization(features)

print(f"Feature shape: {features.shape}")
print(f"PCA explained variance: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}, PC3={explained[2]:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)
axes[0].set_title("Original image", fontweight='bold', fontsize=13)
axes[0].axis('off')

axes[1].imshow(rgb_pca)
axes[1].set_title(f"PCA of patch embeddings (RGB = PC1, PC2, PC3)\n"
                  f"Explained: {explained[0]:.1%} | {explained[1]:.1%} | {explained[2]:.1%}",
                  fontweight='bold', fontsize=11)
axes[1].axis('off')

plt.suptitle("DINOv2 Patch Features — Emergent Semantic Structure",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PCA visualization for ALL images — side by side
fig, axes = plt.subplots(2, len(images), figsize=(16, 8))

for col, (name, img) in enumerate(images.items()):
    feats, _ = extract_patch_features(img, model, processor, DEVICE)
    rgb, expl = pca_rgb_visualization(feats)

    axes[0, col].imshow(img)
    axes[0, col].set_title(name, fontweight='bold', fontsize=12)
    axes[0, col].axis('off')

    axes[1, col].imshow(rgb)
    axes[1, col].set_title(f"PC1={expl[0]:.0%} PC2={expl[1]:.0%} PC3={expl[2]:.0%}",
                           fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel("Original", fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel("PCA Embedding", fontsize=11, fontweight='bold')

plt.suptitle("Emergent Segmentation via PCA of DINOv2 Patch Embeddings",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 3.1 — First PC as a Foreground Mask

Notice that PC1 (the red channel) often separates foreground from background. Let's exploit this.

**Task:**
1. Extract only PC1 from the PCA decomposition
2. Binarize it using Otsu's threshold (from `skimage.filters`)
3. Overlay the resulting mask on the original image
4. Compute the percentage of foreground pixels detected

```python
from skimage.filters import threshold_otsu

# Your code here
```

### **Guided Question 3.2**
> Compare the emergent mask with the ground-truth attention maps from Section 2. Are they consistent? Where do they disagree, and why might that happen?

In [ ]:
# ── Your solution here ───────────────────────────────────────────────────────


---
# Section 4 — Linear Probing: How Good Are the Features?

## The linear probing protocol

To measure the quality of learned representations, we use a standard benchmark: **linear probing**.

1. **Freeze** all DINOv2 weights
2. Extract the **[CLS] feature vector** for each image in a dataset
3. Train only a **linear classifier** (logistic regression) on top
4. Evaluate on a test set

The key insight: if a linear classifier can achieve high accuracy using frozen features, the features must already encode the right semantic information. **No fine-tuning magic needed.**

## Dataset

We'll use **STL-10**, a 10-class image dataset available via `torchvision`. We use a small subset to keep Colab runtime manageable.

| Split | Samples | Classes |
|---|---|---|
| Train (subset) | 1,000 | 10 |
| Test | 800 | 10 |

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import random

random.seed(42)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

print("Downloading STL-10...")
train_dataset = torchvision.datasets.STL10(
    root="./data", split="train", download=True, transform=transform)
test_dataset = torchvision.datasets.STL10(
    root="./data", split="test", download=True, transform=transform)

CLASS_NAMES = ["airplane", "bird", "car", "cat", "deer",
               "dog", "horse", "monkey", "ship", "truck"]

# STL-10 is class-ordered: 500 train and 800 test samples per class.
# Naive range(N) would only cover the first few classes — we need stratified sampling.
N_PER_CLASS_TRAIN = 100   # 100 × 10 = 1000 total
N_PER_CLASS_TEST  =  80   #  80 × 10 =  800 total

def stratified_indices(dataset, n_per_class, n_classes=10):
    labels = [dataset[i][1] for i in range(len(dataset))]
    indices = []
    for cls in range(n_classes):
        cls_idx = [i for i, l in enumerate(labels) if l == cls]
        indices += random.sample(cls_idx, n_per_class)
    return indices

print("Building stratified subsets...")
train_indices = stratified_indices(train_dataset, N_PER_CLASS_TRAIN)
test_indices  = stratified_indices(test_dataset,  N_PER_CLASS_TEST)

train_subset = Subset(train_dataset, train_indices)
test_subset  = Subset(test_dataset,  test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_subset,  batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_subset)} samples ({N_PER_CLASS_TRAIN} per class)")
print(f"Test:  {len(test_subset)} samples ({N_PER_CLASS_TEST} per class)")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# ── Extract [CLS] features for the full dataset ───────────────────────────────
def extract_cls_features(loader, model, device):
    """
    Extracts [CLS] token features from DINOv2 for all images in a DataLoader.
    Returns:
        features : np.ndarray (N, hidden_dim)
        labels   : np.ndarray (N,)
    """
    all_features, all_labels = [], []
    model.eval()

    with torch.no_grad():
        for batch_idx, (images_batch, labels_batch) in enumerate(loader):
            images_batch = images_batch.to(device)

            # DINOv2 expects pixel_values as key
            outputs = model(pixel_values=images_batch)

            # [CLS] token: index 0 of last_hidden_state
            cls_feats = outputs.last_hidden_state[:, 0, :]  # (B, D)
            all_features.append(cls_feats.cpu().numpy())
            all_labels.append(labels_batch.numpy())

            if (batch_idx + 1) % 5 == 0:
                print(f"  Processed {(batch_idx+1) * loader.batch_size} samples...")

    return np.vstack(all_features), np.concatenate(all_labels)


print("Extracting TRAIN features...")
X_train, y_train = extract_cls_features(train_loader, model, DEVICE)

print("Extracting TEST features...")
X_test, y_test = extract_cls_features(test_loader, model, DEVICE)

print(f"\nFeature shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")

In [ ]:
# ── Linear probing with Logistic Regression ───────────────────────────────────

# Normalize features (standard practice)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Training linear probe (logistic regression)...")
clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    random_state=42,
    solver="lbfgs",
    multi_class="multinomial"
)
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)
acc_dinov2 = accuracy_score(y_test, y_pred)

print(f"\n{'='*40}")
print(f"  DINOv2-small Linear Probe")
print(f"  Top-1 Accuracy: {acc_dinov2:.1%}")
print(f"{'='*40}")
print(f"  Baseline (random): 10.0%")
print(f"  Improvement over random: {(acc_dinov2 - 0.1) / 0.1:.0%} relative gain")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title("DINOv2 Linear Probe — Confusion Matrix",
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Per-class accuracy
per_class = cm.diagonal() / cm.sum(axis=1)
print("\nPer-class accuracy:")
for name, acc in zip(CLASS_NAMES, per_class):
    bar = '█' * int(acc * 20)
    print(f"  {name:<10} {bar:<20} {acc:.1%}")

---
# Section 5 — DINOv2 vs Supervised ViT: Feature Quality Showdown

## The key experiment

Now we run the **same linear probing protocol** on a supervised ViT (ViT-Base, pretrained on ImageNet with full labels) and compare results.

This is the central empirical question:

> **Can self-supervised features match or exceed supervised features on a transfer task?**

### **Guided Question 5.1**
> Before running the experiment, form a hypothesis: which model do you expect to win, and why? Consider what each training objective optimizes for.

In [ ]:
# Load supervised ViT-Base (pretrained on ImageNet with labels)
VIT_MODEL = "google/vit-base-patch16-224"

from transformers import ViTModel, ViTImageProcessor

print(f"Loading supervised ViT: {VIT_MODEL}")
vit_processor = ViTImageProcessor.from_pretrained(VIT_MODEL)
vit_model = ViTModel.from_pretrained(VIT_MODEL)
vit_model = vit_model.to(DEVICE)
vit_model.eval()

vit_params = sum(p.numel() for p in vit_model.parameters()) / 1e6
print(f"  ViT-Base parameters: {vit_params:.1f}M")
print(f"  (DINOv2-small for reference: 21M)")

In [ ]:
from torchvision.datasets import STL10

def extract_vit_features(dataset_split, indices, vit_processor, vit_model, device):
    """
    Extract [CLS] features using the supervised ViT processor.
    Uses the same stratified indices as the DINOv2 feature extraction.
    """
    raw_dataset = STL10(root="./data", split=dataset_split, download=False)
    all_features, all_labels = [], []

    vit_model.eval()
    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_pil, label = raw_dataset[idx]
            inputs = vit_processor(images=img_pil, return_tensors="pt").to(device)
            outputs = vit_model(**inputs)
            cls_feat = outputs.last_hidden_state[0, 0, :].cpu().numpy()
            all_features.append(cls_feat)
            all_labels.append(label)

            if (i + 1) % 200 == 0:
                print(f"  Processed {i+1}/{len(indices)}")

    return np.vstack(all_features), np.array(all_labels)


print("Extracting ViT TRAIN features...")
X_train_vit, y_train_vit = extract_vit_features("train", train_indices, vit_processor, vit_model, DEVICE)

print("Extracting ViT TEST features...")
X_test_vit, y_test_vit = extract_vit_features("test", test_indices, vit_processor, vit_model, DEVICE)

print(f"\nViT feature shapes:")
print(f"  X_train_vit: {X_train_vit.shape}")
print(f"  X_test_vit : {X_test_vit.shape}")

In [ ]:
# Linear probe on ViT features
scaler_vit = StandardScaler()
X_train_vit_scaled = scaler_vit.fit_transform(X_train_vit)
X_test_vit_scaled  = scaler_vit.transform(X_test_vit)

print("Training linear probe on ViT features...")
clf_vit = LogisticRegression(max_iter=1000, C=1.0, random_state=42,
                              solver="lbfgs", multi_class="multinomial")
clf_vit.fit(X_train_vit_scaled, y_train_vit)

y_pred_vit = clf_vit.predict(X_test_vit_scaled)
acc_vit = accuracy_score(y_test_vit, y_pred_vit)

print(f"\n{'='*45}")
print(f"  Supervised ViT-Base Linear Probe")
print(f"  Top-1 Accuracy: {acc_vit:.1%}")
print(f"{'='*45}")

In [ ]:
# ── Final comparison plot ─────────────────────────────────────────────────────
models = ["Random\nBaseline", "Supervised ViT-Base\n(86M params)",
          "DINOv2-small\n(21M params)"]
accuracies = [0.10, acc_vit, acc_dinov2]
colors = ['#cccccc', '#4C72B0', '#DD8452']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, [a * 100 for a in accuracies], color=colors,
              edgecolor='black', linewidth=0.8, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{acc:.1%}", ha='center', va='bottom',
            fontsize=13, fontweight='bold')

ax.set_ylim(0, 105)
ax.set_ylabel("Linear Probe Top-1 Accuracy (%)", fontsize=12)
ax.set_title("Feature Quality Comparison: Self-Supervised vs Supervised ViT\n"
             "(STL-10, 1000 train samples, 800 test samples)",
             fontsize=13, fontweight='bold')
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='Chance')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Results Summary")
print(f"   Random baseline : 10.0%")
print(f"   Supervised ViT  : {acc_vit:.1%}")
print(f"   DINOv2-small    : {acc_dinov2:.1%}")

winner = "DINOv2" if acc_dinov2 > acc_vit else "Supervised ViT"
margin = abs(acc_dinov2 - acc_vit)
print(f"\n   🏆 Winner: {winner} by {margin:.1%}")

### **Guided Question 5.2 — Interpreting the Results**

Consider the following:

1. **DINOv2-small has 4× fewer parameters than ViT-Base.** What does it mean if DINOv2 still matches or beats ViT on this linear probe?

2. The linear probe uses **only 1,000 training samples**, while the original ImageNet supervised training used ~1.2M images. What does this tell us about the **sample efficiency** of self-supervised features?

3. **What would full fine-tuning change?** Would the gap between the two models likely increase or decrease, and why?

4. ViT achieves very good performance in STL compared to DINOv2, because the dataset is very similar to Imagenet. You can try other datasets such as EuroSAT available in torch.datasets and check if there is a significant result.

---
# Section 6 — UMAP Visualization of the Embedding Space

As a final exploration, let's project the high-dimensional [CLS] features into 2D using **UMAP** and see how well the classes cluster — without any classifier.

A well-structured embedding space (where same-class samples cluster together) is a prerequisite for good linear separability.

In [ ]:
import umap

PALETTE = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4'
]

def plot_umap(features, labels, title, class_names, ax):
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embedding = reducer.fit_transform(features)

    for cls_idx, (cls_name, color) in enumerate(zip(class_names, PALETTE)):
        mask = labels == cls_idx
        ax.scatter(embedding[mask, 0], embedding[mask, 1],
                   c=color, label=cls_name, alpha=0.6, s=10)

    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(markerscale=2, fontsize=7, loc='upper right',
              framealpha=0.8, ncol=2)


fig, axes = plt.subplots(1, 2, figsize=(16, 7))

print("Computing UMAP for DINOv2 features...")
plot_umap(X_test_scaled, y_test,
          f"DINOv2-small\n(Linear probe acc: {acc_dinov2:.1%})",
          CLASS_NAMES, axes[0])

print("Computing UMAP for Supervised ViT features...")
plot_umap(X_test_vit_scaled, y_test_vit,
          f"Supervised ViT-Base\n(Linear probe acc: {acc_vit:.1%})",
          CLASS_NAMES, axes[1])

plt.suptitle("UMAP Projection of [CLS] Features — Test Set",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Tighter, more separated clusters → better linear separability → higher probe accuracy")

---
# Final Exercise — Your Own Investigation

Choose **one** of the following extensions and implement it. Be prepared to present your findings:

### Option A — Few-shot Generalization
Vary the number of training samples (5, 10, 50, 100, 500, 1000) for both models and plot the **learning curve**. At what sample count does DINOv2 surpass supervised ViT? What does this tell you about representation quality?


### Option B — k-NN Classification
Instead of a linear probe, use **k-nearest neighbors** (varying k from 1 to 50) on the DINOv2 features. Plot accuracy vs k. k-NN requires no training at all — how well does it work? Compare with the linear probe.

In [ ]:
# ── Your final exercise here ─────────────────────────────────────────────────




---

## References

- Caron et al. (2021). *Emerging Properties in Self-Supervised Vision Transformers.* ICCV 2021. [DINO]
- Oquab et al. (2023). *DINOv2: Learning Robust Visual Features without Supervision.* TMLR 2024.
- Dosovitskiy et al. (2020). *An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale.* ICLR 2021.
- HuggingFace DINOv2 model card: https://huggingface.co/facebook/dinov2-small